# Mini-Solveit

In [ ]:
import json
import fasthtml.components as fh

from claudette import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
from toolslm.shell import get_shell

In [ ]:
def get_preview(app): return partial(HTMX, app=app, host=None, port=None)

In [ ]:
def mk_compfn(name, compcls, tag=None):
    if not tag: tag = name
    compfunc = getattr(fh, tag)
    def fn(*c, xtra= '', cls='', **kw):
        xtra = " ".join(f"{compcls}-{o}" for o in xtra.split())
        return compfunc(*c, cls=f'{compcls} {xtra} {cls}', **kw)
    fn.__name__ = name
    globals()[name] = fn



In [ ]:
mk_compfn('Button', 'btn')
mk_compfn('ChatC', 'chat', 'Div') # Call it ChatC as to not confuse with claudette Chat class
mk_compfn('Bubble', 'chat-bubble', 'Div')
mk_compfn('Input', 'input')
mk_compfn('TabInput', 'tab', 'Input')

### Set up the app and start server

In [ ]:
daisy_hdrs = (
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5', rel='stylesheet', type='text/css'),
    Script(src='https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4')
)

app = FastHTML(hdrs=daisy_hdrs, session_cookie="my_cookie")
rt = app.route

In [ ]:
p = get_preview(app)

In [ ]:
srv = JupyUvi(app)

## New features

- Different tabs for code submission and prompt submission
- Build route for prompt submission `execprompt`
- Track history so LLM sees everything the human does
- Multi-user support via session cookies
- Database backend
- Run all button

In [ ]:
user_chats = {}

In [ ]:
msgs = []

In [ ]:
code_history = []

In [ ]:
def qa_bubble(q, a=None):
    return Div(
        ChatC(Bubble(str(q)), xtra="start"),
        ChatC(Bubble(str(a), xtra="primary"), xtra="end") if a else None
    )

p(qa_bubble('2+2', '4'))


HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_-xqbsVuRTqGmM4Z5B3nk9w&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;
      &lt;div class=&quot;chat chat-start &quot;&gt;
        &lt;div class=&quot;chat-bubble  &quot;&gt;2+2&lt;/div&gt;
      &lt;/div&gt;
      &lt;div class=&quot;chat chat-end &quot;&gt;
        &lt;div class=&quot;chat-bubble chat-bubble-primary &quot;&gt;4&lt;/div&gt;
      &lt;/div&gt;
    &lt;/div&gt;
  &lt;/body&gt;
&lt;/html&gt;
" style="width: 100%; height: auto; border: none;" onload="{
        let frame = this;
        window.addEventListener('message', function(e) {
            if (e.source !== frame.contentWindow) return; // Only proceed if the message is from this iframe
            if (e.data.height) frame.style.height = (e.data.height+1) + 'px';
        }, false);
    }" allow="accelerometer; autoplay; camera; clipboard-read; clipboard-write; display-capture; encrypted-media; fullscreen; gamepad; geolocation; gyroscope; hid; identity-credentials-get; idle-detection; magnetometer; microphone; midi; payment; picture-in-picture; publickey-credentials-get; screen-wake-lock; serial; usb; web-share; xr-spatial-tracking"></iframe> )

In [ ]:
def ex(c, sh):
    res = sh.run_cell(c)
    return res.result if res.result is not None else res.stdout

In [ ]:
@rt
def execcode(code:str, dlgid:int, session):
    assert 'uid' in session
    uid = session['uid']
    sh = udlg_active[dlgid]
    
    result = ex(code, sh)
    code_history.append(f'input: {code} -> python:{result}')
    msgs.append({'prompt': code, 'answer': result, 'interaction': 'computer'})
    return Div(qa_bubble(code, result)), Input(placeholder='Enter code here', id='code', hx_swap_oob='outerHTML',cls="flex-1")

In [ ]:
@rt
def execprompt(prompt:str, session):
    global code_history
    assert 'uid' in session
    uid = session['uid']
    c = user_chats[uid]
    
    if code_history:
        hist_prompt = f"code_history: {'/n'.join(code_history)} my_prompt: {prompt}"
        code_history = []
    else:
        hist_prompt = prompt

    c(hist_prompt)
    answer = c.last[0].content[0].text
    msgs.append({'prompt': prompt, 'answer': answer, 'interaction': 'ai'})
    return qa_bubble(prompt, answer), Input(placeholder='Enter prompt here', id='prompt', hx_swap_oob='outerHTML',cls="flex-1")

In [ ]:
@rt
def chatdemo(session):
    if 'uid' not in session:
        session['uid'] = unqid()
    uid = session['uid']
    if uid not in user_chats:
        user_chats[uid] = (get_shell(), Chat(model="claude-haiku-4-5", sp="You are a helpful coding assistant who answers concisely."))

    return Div(cls="max-w-lg")(
        H1("Mini-Solveit", cls="text-3xl text-center font-bold"),
        Div(*[qa_bubble(msg['prompt'], msg['answer']) for msg in msgs], id="dest", cls="overflow-y-auto  max-h-[80vh]"),
        Div(cls="tabs tabs-lift")(
            # Code tab
            TabInput(type="radio", name="mode", aria_label="Code", checked=True),
            Div(cls="tab-content border-base-300 bg-base-100 p-10")(
                Form(hx_get="/execcode", hx_target="#dest", hx_swap="beforeend")(
                    Input(placeholder="Enter code here", id="code", cls="flex-1"),
                    Button("Submit", xtra="primary")
                )
            ),
            # Prompt tab
            TabInput(type="radio", name="mode", aria_label="Prompt"),
            Div(cls="tab-content border-base-300 bg-base-100 p-10")(
                Form(hx_get="/execprompt", hx_target="#dest", hx_swap="beforeend")(
                    Input(placeholder="Enter prompt here", id="prompt", cls="flex-1"),
                    Button("Submit", xtra="primary")
                )
            ),
        )
    )
p(chatdemo)

HTML(<iframe src="/chatdemo" style="width: 100%; height: auto; border: none;" onload="{
        let frame = this;
        window.addEventListener('message', function(e) {
            if (e.source !== frame.contentWindow) return; // Only proceed if the message is from this iframe
            if (e.data.height) frame.style.height = (e.data.height+1) + 'px';
        }, false);
    }" allow="accelerometer; autoplay; camera; clipboard-read; clipboard-write; display-capture; encrypted-media; fullscreen; gamepad; geolocation; gyroscope; hid; identity-credentials-get; idle-detection; magnetometer; microphone; midi; payment; picture-in-picture; publickey-credentials-get; screen-wake-lock; serial; usb; web-share; xr-spatial-tracking"></iframe> )

### DB support

In [ ]:
# Clean starter database
db = database('mini-solveit.db')

class User: id:str
class Dialog: id:int; uid:int; messages:str; name:str

users = db.create(User, transform=True)
dialogs = db.create(Dialog, transform=True)
users.delete_where()
dialogs.delete_where()

<Table dialog (id, uid, messages, name)>

In [ ]:
# Add dummy data
uid = "_XoItG2CRSV6d3xKP3NAJYw"
msgs = json.loads('[{"prompt": "1+9", "answer": 10, "interaction": "computer"}, {"prompt": "a=6", "answer": "", "interaction": "computer"}, {"prompt": "What can I do with `a`?", "answer": "# What can you do with `a`?\\n\\nSince `a = 6`, here are some common operations:\\n\\n**Arithmetic:**\\n```python\\na + 5      # 11\\na * 2      # 12\\na ** 2     # 36\\na / 2      # 3.0\\n```\\n\\n**Comparisons:**\\n```python\\na > 5      # True\\na == 6     # True\\na in range(10)  # True\\n```\\n\\n**Type operations:**\\n```python\\nstr(a)     # \'6\'\\nfloat(a)   # 6.0\\n```\\n\\n**Use in functions:**\\n```python\\nprint(a)\\nlen(str(a))\\nabs(a - 10)\\n```\\n\\n**Reassignment:**\\n```python\\na += 1     # a becomes 7\\na = a * 2  # a becomes 12\\n```\\n\\nWhat specific operation did you want to perform?", "interaction": "ai"}, {"prompt": "a+5", "answer": 11, "interaction": "computer"}, {"prompt": "Btw, my name is Angelika. What is your name?", "answer": "# Nice to meet you, Angelika!\\n\\nI\'m Claude, an AI assistant made by Anthropic. I\'m here to help you with coding questions, debugging, explanations, and more.\\n\\nFeel free to ask me anything about Python or programming! \\ud83d\\ude0a", "interaction": "ai"}]')

u = User(id=uid) # current cookie
u = users.insert(u)
d = dialogs.insert(uid=u.id, messages=json.dumps(msgs), name='my dlg')
users(), dialogs()


([User(id='_XoItG2CRSV6d3xKP3NAJYw')],
 [Dialog(id=1, uid='_XoItG2CRSV6d3xKP3NAJYw', messages='[{"prompt": "1+9", "answer": 10, "interaction": "computer"}, {"prompt": "a=6", "answer": "", "interaction": "computer"}, {"prompt": "What can I do with `a`?", "answer": "# What can you do with `a`?\\n\\nSince `a = 6`, here are some common operations:\\n\\n**Arithmetic:**\\n```python\\na + 5      # 11\\na * 2      # 12\\na ** 2     # 36\\na / 2      # 3.0\\n```\\n\\n**Comparisons:**\\n```python\\na > 5      # True\\na == 6     # True\\na in range(10)  # True\\n```\\n\\n**Type operations:**\\n```python\\nstr(a)     # \'6\'\\nfloat(a)   # 6.0\\n```\\n\\n**Use in functions:**\\n```python\\nprint(a)\\nlen(str(a))\\nabs(a - 10)\\n```\\n\\n**Reassignment:**\\n```python\\na += 1     # a becomes 7\\na = a * 2  # a becomes 12\\n```\\n\\nWhat specific operation did you want to perform?", "interaction": "ai"}, {"prompt": "a+5", "answer": 11, "interaction": "computer"}, {"prompt": "Btw, my name is Angelik

In [ ]:
# u = User(id=unqid())
# u = users.insert(u)


In [ ]:
# users()

In [ ]:
udlg_active = {}

In [ ]:
def run_all(code_msgs):
    sh = get_shell()
    for code in code_msgs: ex(code, sh)
    return sh

In [ ]:
@rt
def showdlg(session, dlgid:int):
    if 'uid' not in session:
        session['uid'] = unqid()
    uid = session['uid']
    if uid not in user_chats: # TODO: This route assumes dialog exists
        user_chats[uid] = Chat(model="claude-haiku-4-5", sp="You are a helpful coding assistant who answers concisely.")

    
    msgs = json.loads(dialogs[dlgid].messages)
    code_msgs = [msg['prompt'] for msg in msgs if msg['interaction']=='computer']
    if dlgid not in udlg_active: udlg_active[dlgid] = run_all(code_msgs)

    return Div(cls="max-w-lg")(
        H1("Mini-Solveit", cls="text-3xl text-center font-bold"),
        Div(*[qa_bubble(msg['prompt'], msg['answer']) for msg in msgs], id="dest", cls="overflow-y-auto  max-h-[80vh]"),
        Div(cls="tabs tabs-lift")(
            # Code tab
            TabInput(type="radio", name="mode", aria_label="Code", checked=True),
            Div(cls="tab-content border-base-300 bg-base-100 p-10")(
                Form(hx_get="/execcode", hx_target="#dest", hx_swap="beforeend")(
                    Hidden(value=dlgid, name="dlgid"),
                    Input(placeholder="Enter code here", id="code", cls="flex-1"),
                    Button("Submit", xtra="primary")
                )
            ),
            # Prompt tab
            TabInput(type="radio", name="mode", aria_label="Prompt"),
            Div(cls="tab-content border-base-300 bg-base-100 p-10")(
                Form(hx_get="/execprompt", hx_target="#dest", hx_swap="beforeend")(
                    Input(placeholder="Enter prompt here", id="prompt", cls="flex-1"),
                    Button("Submit", xtra="primary")
                )
            ),
        )
    )

In [ ]:
p(showdlg.to(dlgid=1))


HTML(<iframe src="/showdlg?dlgid=1" style="width: 100%; height: auto; border: none;" onload="{
        let frame = this;
        window.addEventListener('message', function(e) {
            if (e.source !== frame.contentWindow) return; // Only proceed if the message is from this iframe
            if (e.data.height) frame.style.height = (e.data.height+1) + 'px';
        }, false);
    }" allow="accelerometer; autoplay; camera; clipboard-read; clipboard-write; display-capture; encrypted-media; fullscreen; gamepad; geolocation; gyroscope; hid; identity-credentials-get; idle-detection; magnetometer; microphone; midi; payment; picture-in-picture; publickey-credentials-get; screen-wake-lock; serial; usb; web-share; xr-spatial-tracking"></iframe> )

In [ ]:
udlg_active

{1: <IPython.terminal.interactiveshell.TerminalInteractiveShell at 0x7d0671e7fbf0>}

Every time when I open a new dialog I want to go and see if this dialog already has an active Python shell and if not create a new one and then run all code messages that are already in the history. Now I'm trying to figure out where to fetch and supply the Python shell. Should I pass in the she as an argument when I run the `execcode` route? Should I add a parameter of `dlgid` to the `execcode` route so that this route can look up the active shell in user dialogs dictionary? Any other design ideas?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great question! Here are three design approaches to consider:

1. **Pass `dlgid` to `execcode`** — the route looks up the shell in `udlg_active[dlgid]`. Clean and explicit, but requires the form to know the `dlgid`.

2. **Store `dlgid` in the session** — when `showdlg` is called, save the active `dlgid` to the session. Then `execcode` can silently look it up via `session['dlgid']`. No form changes needed.

3. **Store the shell in the session (by uid)** — instead of keying by `dlgid`, key `udlg_active` by `uid`. Since a user only has one active dialog at a time, this could simplify things.

Which of these feels most natural to you given your current code structure?

<details class='token-usage-details'><summary>$0.0783</summary>

`total=14,542 | in=14,343 | out=199 | cached=0.0% | cache_new=8,266 | $0.0783`

</details>


A user will be allowed to have multiple dialogs. So I think option two and three wouldn't work then.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good point! If a user can have multiple dialogs open, keying by `uid` breaks down. So **option 1 — passing `dlgid` to `execcode`** is the natural fit.

You could add a hidden input to the code form, like:

```python
Hidden(value=dlgid, name="dlgid")
```

Then `execcode` just does `udlg_active[dlgid]` to get the right shell. Does that make sense?

<details class='token-usage-details'><summary>$0.0418</summary>

`total=14,736 | in=14,622 | out=114 | cached=56.5% | cache_new=6,126 | $0.0418`

</details>


Tell me more about hidden inputs. I'm not familiar with those. When should we use them?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Hidden inputs are HTML form fields that are invisible to the user but get submitted along with the form data. They're useful when you need to pass extra information to the server that the user doesn't need to see or interact with.

A classic example: you have a form to edit a record, and the server needs to know *which* record — but you don't want the user editing the ID field. You just tuck it away as a hidden input.

In your case, the form needs to tell `/execcode` which dialog it belongs to, but there's no reason to show the user a "dialog ID" field — so a hidden input is perfect.

Does that make sense? Want to try adding one to your code form?

<details class='token-usage-details'><summary>$0.0121</summary>

`total=14,967 | in=14,811 | out=156 | cached=97.1% | cache_new=194 | $0.0121`

</details>


Please give me a code snippet of how I can add the hidden input to my `execcode` route.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Sure! In the `showdlg` route, update the code form to include a hidden input with the `dlgid`:

```python
Form(hx_get="/execcode", hx_target="#dest", hx_swap="beforeend")(
    Hidden(value=dlgid, name="dlgid"),
    Input(placeholder="Enter code here", id="code", cls="flex-1"),
    Button("Submit", xtra="primary")
)
```

And then update `execcode` to accept and use it:

```python
@rt
def execcode(code: str, dlgid: int, session):
    sh = udlg_active[dlgid]
    ...
```

Give that a try and let me know how it goes!

<details class='token-usage-details'><summary>$0.0135</summary>

`total=15,248 | in=15,043 | out=205 | cached=96.9% | cache_new=231 | $0.0135`

</details>


Is there a way for me to store the Python shell and claudette Chat (get_shell(), Chat()) inside the dialogs table? If yes, what datatype would it be?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Short answer: not directly — those are **live Python objects** that can't be straightforwardly serialized to a database.

Here's why, and what you *can* store:

- **`get_shell()`** holds a running IPython kernel with live state (variables, imports, etc.). There's no standard way to serialize that.
- **`Chat()`** holds conversation history (`c.h`), but the object itself has methods and state that can't be pickled to SQLite easily.

**What you *can* store:**

| Thing | Storable? | How |
|---|---|---|
| Chat history (`c.h`) | ✅ | `json.dumps(c.h)` |
| Code history | ✅ | Already doing this with `msgs` |
| Shell state | ❌ | Not feasible |

So the practical approach is to **store only the chat history** in the DB, and **reconstruct** a fresh `Chat` object from it when a dialog is loaded — which you can do with claudette's `h` parameter or by replaying history.

Would you like to explore how to reconstruct a `Chat` from saved history?

<details class='token-usage-details'><summary>$0.0916</summary>

`total=17,084 | in=16,784 | out=300 | cached=0.0% | cache_new=8,266 | $0.0916`

</details>


Sounds like it might be better if I adapt my app so that it can rerun all the code cells when it loads a new dialog. What you do think about that idea? Or any other suggestions?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's actually a solid idea, and it's exactly how solveit itself works — it has a "Run all" button that re-executes code top to bottom when you reload.

A few thoughts on tradeoffs:

- ✅ **Simple & reliable** — no serialization headaches, shell state is always fresh and consistent
- ✅ **Idempotent** — replaying code is predictable
- ⚠️ **Side effects** — if code has side effects (file writes, API calls, etc.) re-running could be problematic
- ⚠️ **Speed** — for long sessions with slow code, rerunning everything could be slow

An alternative worth considering: **store code and prompt messages separately**, so you know exactly which cells to re-execute vs. which are just display.

Which of these concerns applies most to your use case?

<details class='token-usage-details'><summary>$0.0586</summary>

`total=17,383 | in=17,153 | out=230 | cached=48.2% | cache_new=8,638 | $0.0586`

</details>


I am already tracking if an interaction was from a computer or an AI. I think that should help us figure out which messages to re-execute. Okay let me think through how I could implement a "Run all" button. I would have to get a fresh shell. Then a list of all the code messages, and run them through the shell one-by-one. Does that sounds right?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, that's exactly right! Your existing `interaction` field makes it straightforward to filter just the `'computer'` messages.

One small thing to think about: each code cell's output might have depended on the *previous* cell's state, so order matters. Are your messages already stored in order?

<details class='token-usage-details'><summary>$0.0121</summary>

`total=17,558 | in=17,489 | out=69 | cached=96.6% | cache_new=298 | $0.0121`

</details>


Yes I append to them to a list when they got originally executed, so they should be in order.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Perfect, that gives you everything you need then! So your "Run all" logic would be roughly:

1. Get a fresh shell
2. Filter `msgs` for `interaction == 'computer'`
3. Loop through them in order, passing each `prompt` to `ex()`

Give it a go — how would you start writing that?

<details class='token-usage-details'><summary>$0.0117</summary>

`total=17,713 | in=17,634 | out=79 | cached=97.5% | cache_new=206 | $0.0117`

</details>


In [ ]:
m = json.dumps([{'prompt': '1+9', 'answer': 10, 'interaction': 'computer'},
 {'prompt': 'a=6', 'answer': '', 'interaction': 'computer'},
 {'prompt': 'What can I do with `a`?',
  'answer': "# What can you do with `a`?\n\nSince `a = 6`, here are some common operations:\n\n**Arithmetic:**\n```python\na + 5      # 11\na * 2      # 12\na ** 2     # 36\na / 2      # 3.0\n```\n\n**Comparisons:**\n```python\na > 5      # True\na == 6     # True\na in range(10)  # True\n```\n\n**Type operations:**\n```python\nstr(a)     # '6'\nfloat(a)   # 6.0\n```\n\n**Use in functions:**\n```python\nprint(a)\nlen(str(a))\nabs(a - 10)\n```\n\n**Reassignment:**\n```python\na += 1     # a becomes 7\na = a * 2  # a becomes 12\n```\n\nWhat specific operation did you want to perform?",
  'interaction': 'ai'},
 {'prompt': 'a+5', 'answer': 11, 'interaction': 'computer'},
 {'prompt': 'Btw, my name is Angelika. What is your name?',
  'answer': "# Nice to meet you, Angelika!\n\nI'm Claude, an AI assistant made by Anthropic. I'm here to help you with coding questions, debugging, explanations, and more.\n\nFeel free to ask me anything about Python or programming! 😊",
  'interaction': 'ai'}])
m

'[{"prompt": "1+9", "answer": 10, "interaction": "computer"}, {"prompt": "a=6", "answer": "", "interaction": "computer"}, {"prompt": "What can I do with `a`?", "answer": "# What can you do with `a`?\\n\\nSince `a = 6`, here are some common operations:\\n\\n**Arithmetic:**\\n```python\\na + 5      # 11\\na * 2      # 12\\na ** 2     # 36\\na / 2      # 3.0\\n```\\n\\n**Comparisons:**\\n```python\\na > 5      # True\\na == 6     # True\\na in range(10)  # True\\n```\\n\\n**Type operations:**\\n```python\\nstr(a)     # \'6\'\\nfloat(a)   # 6.0\\n```\\n\\n**Use in functions:**\\n```python\\nprint(a)\\nlen(str(a))\\nabs(a - 10)\\n```\\n\\n**Reassignment:**\\n```python\\na += 1     # a becomes 7\\na = a * 2  # a becomes 12\\n```\\n\\nWhat specific operation did you want to perform?", "interaction": "ai"}, {"prompt": "a+5", "answer": 11, "interaction": "computer"}, {"prompt": "Btw, my name is Angelika. What is your name?", "answer": "# Nice to meet you, Angelika!\\n\\nI\'m Claude, an AI assis

In [ ]:
msgs = json.loads(m)
msgs

[{'prompt': '1+9', 'answer': 10, 'interaction': 'computer'},
 {'prompt': 'a=6', 'answer': '', 'interaction': 'computer'},
 {'prompt': 'What can I do with `a`?',
  'answer': "# What can you do with `a`?\n\nSince `a = 6`, here are some common operations:\n\n**Arithmetic:**\n```python\na + 5      # 11\na * 2      # 12\na ** 2     # 36\na / 2      # 3.0\n```\n\n**Comparisons:**\n```python\na > 5      # True\na == 6     # True\na in range(10)  # True\n```\n\n**Type operations:**\n```python\nstr(a)     # '6'\nfloat(a)   # 6.0\n```\n\n**Use in functions:**\n```python\nprint(a)\nlen(str(a))\nabs(a - 10)\n```\n\n**Reassignment:**\n```python\na += 1     # a becomes 7\na = a * 2  # a becomes 12\n```\n\nWhat specific operation did you want to perform?",
  'interaction': 'ai'},
 {'prompt': 'a+5', 'answer': 11, 'interaction': 'computer'},
 {'prompt': 'Btw, my name is Angelika. What is your name?',
  'answer': "# Nice to meet you, Angelika!\n\nI'm Claude, an AI assistant made by Anthropic. I'm here t

In [ ]:
code_msgs = [msg['prompt'] for msg in msgs if msg['interaction']=='computer']
code_msgs

['1+9', 'a=6', 'a+5']

In [ ]:
sh = get_shell()

In [ ]:
sh = run_all(code_msgs)

In [ ]:
ex('a', sh)

6

In [ ]:
@rt
def dlglist(session):
    if 'uid' not in session:
        session['uid'] = unqid()
    uid = session['uid']
    try: user = users[uid]
    except NotFoundError: user = users.insert(id=uid)
    udlgs = dialogs('uid=?', [user.id])
    return Ul(*[Li(A(dlg.name, href=showdlg.to(dlgid=dlg.id), cls="link"), cls="list-row") for dlg in udlgs], cls="list")

In [ ]:
p(dlglist)

HTML(<iframe src="/dlglist" style="width: 100%; height: auto; border: none;" onload="{
        let frame = this;
        window.addEventListener('message', function(e) {
            if (e.source !== frame.contentWindow) return; // Only proceed if the message is from this iframe
            if (e.data.height) frame.style.height = (e.data.height+1) + 'px';
        }, false);
    }" allow="accelerometer; autoplay; camera; clipboard-read; clipboard-write; display-capture; encrypted-media; fullscreen; gamepad; geolocation; gyroscope; hid; identity-credentials-get; idle-detection; magnetometer; microphone; midi; payment; picture-in-picture; publickey-credentials-get; screen-wake-lock; serial; usb; web-share; xr-spatial-tracking"></iframe> )

In [ ]:
!ls -l

total 304
-rw------- 1 solveit solveit 104766 Apr 15 07:39  Cookies.ipynb
-rw------- 1 solveit solveit 105407 Apr 15 07:39  Cookies_with_my_notes.ipynb
-rw------- 1 solveit solveit  20289 Apr 16 08:36 'Extend Mini-Solveit Multi-User.ipynb'
-rw------- 1 solveit solveit    130 Apr  8 12:32 'Learn about Groq.ipynb'
-rw-r--r-- 1 solveit solveit  16384 Apr 16 08:31  mini-solveit.db
-rw-r--r-- 1 solveit solveit  32768 Apr 16 08:36  mini-solveit.db-shm
-rw-r--r-- 1 solveit solveit  20632 Apr 16 08:36  mini-solveit.db-wal


**NEXT**: Work with Claude (Opus) to see if I can refactor things. Also if there is a nicer way to deal with the global variable, or if some decoupling would help there.

Then, [51:30] learn about session cookies.

[1:03:20] What is dependency injection? Related to the sesson cookie..

- I think I need to add the shell and Chat object that belong to a certain dialog to the database.
- add dialog with box to insert name, and create
- back button from the dialog list

In [ ]:
import base64


In [ ]:
cookie_text = "eyJ1aWQiOiAiX1hvSXRHMkNSU1Y2ZDN4S1AzTkFKWXcifQ==.aeCwYQ.STte4YdnYEf3biL3XrzVs5M9SGE".split('.')[0]
base64.b64decode(cookie_text)

b'{"uid": "_XoItG2CRSV6d3xKP3NAJYw"}'